#Libraries

In [ ]:
import numpy as np
import pandas as pd

import seaborn as sns
import tensorflow as tf
import random
import os
import time

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.models import load_model
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_squared_error, mean_absolute_error,mean_absolute_percentage_error, r2_score

import matplotlib.pyplot as plt

In [ ]:
print("GPU available :", tf.config.list_physical_devices('GPU'))

# fix the seed

In [ ]:
seed_value = 42
os.environ['PYTHONHASHSEED'] = str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

#dataset







---



**1.   Data reading**

In [ ]:
df = pd.read_csv('Timeseries_32.207_-6.532_SA3_300kWp_crystSi_18_33deg_0deg_2010_2023.csv',
                 skiprows=10,
                 nrows=122723)
df = df.iloc[78888:].copy()
print(f"Forme du tableau : {df.shape}")
print(df.head())
# print("Detected columns :", df.columns.tolist())
# print(df.head())
print(df.tail())



2.   **Feature Engineering**



*   Temporal cycling







In [ ]:
# datetime
df['time'] = pd.to_datetime(df['time'], format='%Y%m%d:%H%M')# Convert to datetime format
df['hour'] = df['time'].dt.hour  # Extract hour (0 to 23)
df['month'] = df['time'].dt.month  # Extract month (1 to 12) for seasonality
# Selecting columns for the model
features = ['P', 'Gb(i)', 'Gd(i)', 'T2m', 'WS10m', 'hour', 'month']
data = df[features].values
# Hour Cycling (24h period)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
# Month Cycling (12 months period)
df['month_sin'] = np.sin(2 * np.pi * (df['month']-1) / 12)
df['month_cos'] = np.cos(2 * np.pi * (df['month']-1) / 12)

# We define the columns that we want to keep for the model
# Note: We keep Gb(i) and Gd(i) because they are crucial for the solar
features_finales = [
    'P', 'Gb(i)', 'Gd(i)', 'T2m', 'WS10m',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos'
]

# We create the working dataset
df_final = df[features_finales]

print("Preview of the final dataset (9 columns):")
print(df_final.head())



*   Visualization


In [ ]:
# We trace hour_sin and hour_cos over the first 48 hours
plt.figure(figsize=(12, 4))
plt.plot(df['hour_sin'][:48], label='Sinus Heure')
plt.plot(df['hour_cos'][:48], label='Cosinus Heure')
plt.title("Temporal Cycling Verification (48h)")
plt.legend()
plt.show()



*   Standardization



In [ ]:
scaler = MinMaxScaler(feature_range=(-1, 1))
data_scaled = scaler.fit_transform(df_final)



*   Creation of Matrices (3D)


In [ ]:
def create_sequences(data, window_size):
    X = []
    y = []

    # We iterate through the dataset
    for i in range(len(data) - window_size):
        # The X sequence: the past 24 hours (all 9 columns)
        # From index i up to i + window_size (exclusive)
        X.append(data[i : i + window_size, :])

        # The Target y: Power P (column 0) at time t + window_size
        # This is exactly the value that follows the sequence
        y.append(data[i + window_size, 0])

    return np.array(X), np.array(y)

# Application on your normalized Afourer data
X, y = create_sequences(data_scaled, window_size=24)



*   Slicing




In [ ]:
split_idx = int(len(X) * 0.8)

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

X_test = X_test[-8760:]
y_test = y_test[-8760:]

y_train = y_train.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)

print(f"Ready for training! X_Train size: {X_train.shape}")
print(f"Ready for training! y_Train size: {y_train.shape}")
print(f"Ready for testing! X_test size: {X_test.shape}")
print(f"Ready for testing! y_test size: {y_test.shape}")

# GRU Mdel



---





**1.   Architecture**


In [ ]:
model_gru = Sequential([
    # GRU layer with 32 units, input_shape = (24 timesteps, 9 variables)
    GRU(32, input_shape=(24, 9), activation='tanh', return_sequences=False),
    Dropout(0.2),
    Dense(1) # Output layer: 1 neuron to predict Power P
])

# We use the 'adam' optimizer (standard) and 'mse' (Mean Squared Error) loss
model_gru.compile(optimizer='adam', loss='mse', metrics=['mae'])

model_gru.summary()






**2.  Training**





In [ ]:
start_time = time.time()
history_gru = model_gru.fit(
    X_train, y_train,
    epochs=50,             # Number of passes over the data
    batch_size=32,          # Data group size per weight update
    validation_split=0.1,   # We keep 10% of the training data to monitor learning
    shuffle=True,           # SHUFFLE sequences for stability
    verbose=1
)
end_time = time.time()

# Calculate and display the result
training_duration = end_time - start_time
print(f"--- Total training time: {training_duration:.2f} seconds ---")



**3.  Visualization**



In [ ]:
def plot_learning_curves(history):
    # Retrieving history data
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(loss) + 1)

    # Creating the graph
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, loss, 'bo-', label='Training Error (Loss)')
    plt.plot(epochs, val_loss, 'ro-', label='Validation Error (Val_loss)')

    plt.title('Learning Curves - GRU Model (Afourer)')
    plt.xlabel('Epochs')
    plt.ylabel('Error (MSE)')
    plt.legend()
    plt.grid(True)
    plt.show()

    return loss, val_loss

In [ ]:
loss, val_loss = plot_learning_curves(history_gru)

In [ ]:
loss, val_loss

In [ ]:
y_pred_scaled = model_gru.predict(X_test)
# Denormalization function (Dummy Matrix Method)
def denormalize(scaled_values, scaler, n_features=9):
    # Creating an empty matrix of 9 columns (the number of original columns)
    dummy = np.zeros((len(scaled_values), n_features))
    # We place our (scaled) values in the first column (Power P)
    dummy[:, 0] = scaled_values.flatten()
    # We apply the inverse of the MinMaxScaler
    inverse = scaler.inverse_transform(dummy)
    # We only retrieve the first column (the actual power)
    return inverse[:, 0]

# Conversion to real units (Watts according to my dataset)
y_test_real = denormalize(y_test, scaler)
y_pred_real = denormalize(y_pred_scaled, scaler)

# Calculation of scientific scores
rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
mae = mean_absolute_error(y_test_real, y_pred_real)
r2 = r2_score(y_test_real, y_pred_real)
nMAE = 100 * (mae / np.max(y_test_real))
nRMSE  = 100 * (rmse / np.max(y_test_real))

print(f"--- RESULTS (Window = 24h) ---")
print(f"RMSE : {rmse:.3f}")
print(f"MAE  : {mae:.3f}")
print(f"nRMSE : {nRMSE:.3f}","%")
print(f"nMAE  : {nMAE:.3f}","%")
print(f"R²   : {r2:.4f}")

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(y_test_real[200:368], label='Real Afourer', color='blue', alpha=0.6)
plt.plot(y_pred_real[200:368], label='GRU Prediction', color='red', linestyle='--')
plt.ylabel('Power (Watt)')
plt.title('zoom in on a week of test')
plt.legend()
plt.show()

In [ ]:
y_pred_real[200:368]